In [1]:
import os

In [7]:
# 数据准备
os.makedirs(os.path.join("..", 'data'), exist_ok=True)
data_file = os.path.join('..', 'data', 'house_tiny.csv')
with open(data_file, 'w') as f:
    f.write('NumRooms,Alley,Price\n')
    f.write('NA,Pave,127500\n')
    f.write('2,NA,106000\n')
    f.write('4,NA,178100\n')
    f.write('NA,NA,140000\n')

In [61]:
import pandas as pd
data = pd.read_csv(data_file)
print(data)

   NumRooms Alley   Price
0       NaN  Pave  127500
1       2.0   NaN  106000
2       4.0   NaN  178100
3       NaN   NaN  140000


## 缺失数据处理：插值和删除

In [62]:
# 插值:处理数值型，用平均值填充
inputs, outputs = data.iloc[:, 0:2], data.iloc[:, 2]

inputs = inputs.fillna(inputs.mean(numeric_only=True))
print(inputs)

   NumRooms Alley
0       3.0  Pave
1       2.0   NaN
2       4.0   NaN
3       3.0   NaN


In [63]:
# 非数值型：转换为类别
inputs = pd.get_dummies(inputs, dummy_na=True)# dummy_na = True NaN也做一种类别
print(type(inputs))
print(inputs)
print(inputs.dtypes)
inputs = inputs.applymap(float)
print(inputs.dtypes)
import torch
x = torch.tensor(inputs.values)
x

<class 'pandas.core.frame.DataFrame'>
   NumRooms  Alley_Pave  Alley_nan
0       3.0        True      False
1       2.0       False       True
2       4.0       False       True
3       3.0       False       True
NumRooms      float64
Alley_Pave       bool
Alley_nan        bool
dtype: object
NumRooms      float64
Alley_Pave    float64
Alley_nan     float64
dtype: object


C:\user\default\ipykernel_17504\1188276244.py:6: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  inputs = inputs.applymap(float)


tensor([[3., 1., 0.],
        [2., 0., 1.],
        [4., 0., 1.],
        [3., 0., 1.]], dtype=torch.float64)

In [2]:
import torch
x = torch.randn((2,5,4))
x


tensor([[[ 0.1543,  2.1921,  0.2766,  0.4810],
         [-0.1446, -1.6595, -0.9477, -0.3769],
         [ 0.3390,  0.7904, -1.0637, -0.3176],
         [ 0.1695, -1.6695,  0.9467,  0.0550],
         [ 1.7197, -0.6130, -0.1610,  1.3835]],

        [[ 0.9063,  1.2957,  2.5932, -1.1891],
         [-0.5174, -1.6850,  0.8415, -0.4854],
         [-1.2997,  1.1312, -0.5680, -0.6650],
         [ 1.1833,  1.8882,  0.5648, -0.4129],
         [-0.7845,  0.4095, -3.4813,  0.7258]]])

In [4]:
x.sum(axis=0)

tensor([[ 1.0606,  3.4878,  2.8697, -0.7082],
        [-0.6620, -3.3445, -0.1062, -0.8623],
        [-0.9607,  1.9217, -1.6318, -0.9826],
        [ 1.3528,  0.2188,  1.5115, -0.3580],
        [ 0.9352, -0.2035, -3.6423,  2.1092]])

In [5]:
x.sum(axis=1)

tensor([[ 2.2379, -0.9594, -0.9492,  1.2250],
        [-0.5119,  3.0396, -0.0498, -2.0267]])

In [6]:
x.sum(axis=2)

tensor([[ 3.1039, -3.1286, -0.2519, -0.4983,  2.3292],
        [ 3.6061, -1.8463, -1.4014,  3.2234, -3.1306]])

In [7]:
x.sum(axis=(0,2), keepdims=True)

tensor([[[ 6.7099],
         [-4.9749],
         [-1.6533],
         [ 2.7251],
         [-0.8014]]])

# 自动求导实现

In [8]:
import torch
x = torch.arange(4.0)
x

tensor([0., 1., 2., 3.])

In [9]:
x.requires_grad_(True)# pytorch记录梯度
x.grad

In [10]:
y = 2 * torch.dot(x,x)
y

tensor(28., grad_fn=<MulBackward0>)

In [11]:
y.backward()# 求解grad需要手动backward
x.grad

tensor([ 0.,  4.,  8., 12.])

In [12]:
# 在默认情况下，pytorch会累积梯度，我们需要清除之前的值
x.grad.zero_()
y = x.sum()
y.backward()
x.grad

tensor([1., 1., 1., 1.])

In [13]:
x.grad.zero_()
y = x * x
sum().backward() # 在深度学习中一般不会对矩阵求导，一般是对一个标量进行求导，可以联想一下最后的loss是一个标量
x.grad

tensor([0., 2., 4., 6.])

In [14]:
x.grad.zero_()
y = x * x
u = y.detach()# y当成一个常数
z = u * x
z.sum().backward()
x.grad == u

tensor([True, True, True, True])

In [15]:
x.grad.zero_()
y.sum().backward()
x.grad == 2 * x

tensor([True, True, True, True])

In [18]:
def f(a):
    b = a * 2
    while b.norm() < 1000:
        b = b * 2
    if b.sum() > 0 :
        c = b
    else:
        c = 100 * b
    return c

a = torch.randn(size=(),requires_grad=True)
d = f(a)
d.backward()
a.grad = d/a
a.grad

tensor(2048., grad_fn=<DivBackward0>)